# nnUNet v2 — Pancreatic Cyst Segmentation Pipeline

This notebook documents the full nnUNet v2 workflow for the pancreatic cyst CT segmentation dataset.

## Dataset Summary
| Property | Value |
|---|---|
| Total cases | 358 |
| Train / Val / Test | 247 / 37 / 74 |
| Modality | CT (single channel) |
| Labels | 0 = background, 1 = cyst |
| Spacing (xy) | ~0.78 – 1.37 mm (variable) |
| Spacing (z) | ~3.6 – 7.2 mm (anisotropic) |
| File format | NIfTI (.nii.gz) |

## System Summary
| Resource | Value |
|---|---|
| GPU | NVIDIA RTX 4060 Ti (16 GB VRAM) |
| RAM | 62 GB |
| CPUs | 20 |
| Disk free | ~637 GB |
| CUDA | 12.6 |
| PyTorch | 2.11.0+cu126 |
| nnUNetv2 | 2.6.4 |
| Python | 3.13.2 (`.venv`) |

## Directory Layout
```
nnUnet/
├── nnUNet_raw/
│   └── Dataset001_PancreasCyst/
│       ├── imagesTr/          # 284 train+val images  (CASE_0000.nii.gz)
│       ├── labelsTr/          # 284 masks             (CASE.nii.gz)
│       ├── imagesTs/          # 74  test images
│       └── dataset.json
├── nnUNet_preprocessed/       # written by plan_and_preprocess
├── nnUNet_results/            # written by training
└── nnUNet_pipeline.ipynb      # this notebook
```

---
## 0. Environment Setup

Always activate the project venv and set the three required nnUNet env vars before running any command.

In [ ]:
import os, subprocess, sys

# ── paths ────────────────────────────────────────────────────────────────────
BASE      = "/home/huy/quan_nguyen/aima/pancrea_cyst"
DATA_DIR  = f"{BASE}/data"
VENV_BIN  = f"{BASE}/.venv/bin"
NNUNET    = f"{BASE}/nnUnet"

RAW          = f"{NNUNET}/nnUNet_raw"
PREPROCESSED = f"{NNUNET}/nnUNet_preprocessed"
RESULTS      = f"{NNUNET}/nnUNet_results"

DATASET_ID   = 1          # Dataset001_PancreasCyst
DATASET_NAME = "Dataset001_PancreasCyst"

# ── inject into process env so subprocesses inherit them ─────────────────────
os.environ["nnUNet_raw"]          = RAW
os.environ["nnUNet_preprocessed"] = PREPROCESSED
os.environ["nnUNet_results"]      = RESULTS
os.environ["PATH"]                = VENV_BIN + ":" + os.environ["PATH"]

# ── helper: run a shell command and stream output ─────────────────────────────
def run(cmd, **kwargs):
    """Run cmd in a subprocess, streaming stdout/stderr."""
    env = {**os.environ, **kwargs.pop("env", {})}
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env=env, **kwargs
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {cmd}")

print("nnUNet_raw:",          RAW)
print("nnUNet_preprocessed:", PREPROCESSED)
print("nnUNet_results:",      RESULTS)

---
## 1. Dataset Exploration

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import json, os

# ── dataset.json ──────────────────────────────────────────────────────────────
with open(f"{RAW}/{DATASET_NAME}/dataset.json") as f:
    ds_json = json.load(f)

print("Dataset JSON:")
print(f"  name          : {ds_json['name']}")
print(f"  numTraining   : {ds_json['numTraining']}")
print(f"  channel_names : {ds_json['channel_names']}")
print(f"  labels        : {ds_json['labels']}")

In [ ]:
# ── spacing / shape stats across full dataset ─────────────────────────────────
img_dir = f"{DATA_DIR}/images"
spacings, shapes = [], []

for fname in sorted(os.listdir(img_dir)):
    img = nib.load(f"{img_dir}/{fname}")
    sp  = np.sqrt((img.affine[:3, :3] ** 2).sum(axis=0))
    spacings.append(sp)
    shapes.append(img.shape)

spacings = np.array(spacings)
shapes   = np.array(shapes)

print("Spacing (mm)  min / median / max")
for ax, name in enumerate(["x", "y", "z"]):
    print(f"  {name}: {spacings[:,ax].min():.2f} / {np.median(spacings[:,ax]):.2f} / {spacings[:,ax].max():.2f}")

print("\nShape (voxels) min / median / max")
for ax, name in enumerate(["dim0", "dim1", "dim2"]):
    print(f"  {name}: {shapes[:,ax].min()} / {int(np.median(shapes[:,ax]))} / {shapes[:,ax].max()}")

In [ ]:
# ── visualise a random case ───────────────────────────────────────────────────
import random

sample_img_name = random.choice(sorted(os.listdir(img_dir)))
sample_stem     = sample_img_name.replace(".nii.gz", "")

# find matching mask
import re
def img_stem_to_mask(stem):
    s = re.sub(r'([A-Za-z]+)(\d+)', lambda m: m.group(1).lower()+'_'+m.group(2), stem)
    return f"cyst_{s}.nii.gz"

img  = nib.load(f"{img_dir}/{sample_img_name}").get_fdata()
mask_path = f"{DATA_DIR}/masks/{img_stem_to_mask(sample_stem)}"
msk  = nib.load(mask_path).get_fdata() if os.path.exists(mask_path) else None

mid_z = img.shape[2] // 2
fig, axes = plt.subplots(1, 2 if msk is not None else 1, figsize=(12, 5))
if msk is None:
    axes = [axes]

axes[0].imshow(img[:, :, mid_z].T, cmap="gray", origin="lower", vmin=-200, vmax=400)
axes[0].set_title(f"CT — {sample_stem} (slice {mid_z})")
axes[0].axis("off")

if msk is not None:
    axes[1].imshow(img[:, :, mid_z].T, cmap="gray", origin="lower", vmin=-200, vmax=400)
    axes[1].imshow(msk[:, :, mid_z].T, cmap="Reds", alpha=0.4, origin="lower")
    axes[1].set_title("CT + cyst mask overlay")
    axes[1].axis("off")

plt.tight_layout()
plt.show()
print(f"Cyst voxels: {int(msk.sum()) if msk is not None else 'N/A'} / {img.size} total")

---
## 2. System Check

In [ ]:
import torch, shutil

print("=== GPU ===")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free, total = torch.cuda.mem_get_info(i)
        print(f"  [{i}] {props.name}")
        print(f"      VRAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        print(f"      CUDA capability: {props.major}.{props.minor}")
else:
    print("  No CUDA GPU available — training will use CPU (very slow!)")

print("\n=== RAM ===")
import psutil
vm = psutil.virtual_memory()
print(f"  Total: {vm.total/1e9:.1f} GB | Available: {vm.available/1e9:.1f} GB")

print("\n=== Disk ===")
disk = shutil.disk_usage(NNUNET)
print(f"  Total: {disk.total/1e9:.0f} GB | Free: {disk.free/1e9:.0f} GB")

print("\n=== Sufficiency Assessment ===")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
ram_gb  = vm.total / 1e9
disk_gb = disk.free / 1e9

checks = {
    "GPU VRAM ≥ 8 GB (3D fullres)": vram_gb >= 8,
    "GPU VRAM ≥ 4 GB (2D)": vram_gb >= 4,
    "RAM ≥ 32 GB": ram_gb >= 32,
    "Disk free ≥ 100 GB": disk_gb >= 100,
}
for check, ok in checks.items():
    print(f"  {'✓' if ok else '✗'} {check}")

---
## 3. Preprocessing (plan_and_preprocess)

**Already completed.** Run this cell only if you need to redo it (e.g. after changing the dataset).

### What it does
1. **Fingerprint extraction** — reads all training images, collects spacing, shape, intensity stats  
2. **Experiment planning** — decides architectures, patch sizes, batch sizes for each configuration  
3. **Preprocessing** — resamples + normalises all training cases and saves to `nnUNet_preprocessed/`

### Plans generated (Dataset001)
| Config | Patch size | Batch size | Stages |
|---|---|---|---|
| 2D | 320 × 320 | 32 | 7 |
| 3D fullres | 40 × 192 × 256 | 2 | 6 |
| 3D lowres | skipped (too similar to fullres) | — | — |

> **Note on origin/direction warnings:** The CAD cases have masks exported with zeroed-out NIfTI headers (origin 0,0,0 and identity direction), while images carry the original DICOM metadata. This is a data provenance issue — nnUNet handles it gracefully because it resamples into a canonical space. However, for downstream analysis you may want to copy the image affine onto the masks.

In [ ]:
# ── OPTIONAL: re-run plan_and_preprocess ──────────────────────────────────────
# run(
#     f"nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity -np 8"
# )

In [ ]:
# ── Inspect the generated plans ───────────────────────────────────────────────
import json

plans_path = f"{PREPROCESSED}/{DATASET_NAME}/nnUNetPlans.json"
with open(plans_path) as f:
    plans = json.load(f)

print(f"Plans file: {plans_path}\n")
for cfg_name, cfg in plans["configurations"].items():
    print(f"=== {cfg_name} ===")
    print(f"  patch_size  : {cfg.get('patch_size')}")
    print(f"  batch_size  : {cfg.get('batch_size')}")
    print(f"  spacing     : {cfg.get('spacing')}")
    n_stages = cfg.get('architecture', {}).get('arch_kwargs', {}).get('n_stages', 'N/A')
    print(f"  n_stages    : {n_stages}")
    print()

---
## 4. Training

nnUNet uses **5-fold cross-validation** by default. Each fold trains one model on 4/5 of the data and validates on 1/5. The ensemble of all 5 folds is used for final inference.

### Recommended training order for this dataset
Given the thick-slice anisotropy (z ~4–7 mm vs xy ~1 mm), **3D fullres** is typically the best single config. 2D can also be competitive with anisotropic data. Running both and letting nnUNet pick the best is ideal.

### Training commands
```bash
# Set env vars first
export nnUNet_raw=/home/huy/quan_nguyen/aima/pancrea_cyst/nnUnet/nnUNet_raw
export nnUNet_preprocessed=/home/huy/quan_nguyen/aima/pancrea_cyst/nnUnet/nnUNet_preprocessed
export nnUNet_results=/home/huy/quan_nguyen/aima/pancrea_cyst/nnUnet/nnUNet_results

# 3D fullres — all 5 folds (run in sequence or in parallel on separate GPUs)
for FOLD in 0 1 2 3 4; do
    nnUNetv2_train 1 3d_fullres $FOLD --npz
done

# 2D — all 5 folds
for FOLD in 0 1 2 3 4; do
    nnUNetv2_train 1 2d $FOLD --npz
done
```

### Training a single fold from the notebook

In [ ]:
# ── Train 3D fullres, fold 0 ──────────────────────────────────────────────────
# Uncomment to run. Expect ~12–20 h per fold on RTX 4060 Ti.

CONFIG = "3d_fullres"   # or "2d"
FOLD   = 0

# run(f"nnUNetv2_train {DATASET_ID} {CONFIG} {FOLD} --npz")

In [ ]:
# ── Monitor training progress ─────────────────────────────────────────────────
import os, json

CONFIG = "3d_fullres"
FOLD   = 0

log_path = (
    f"{RESULTS}/{DATASET_NAME}/nnUNetTrainer__nnUNetPlans__{CONFIG}/"
    f"fold_{FOLD}/training_log_*.txt"
)
import glob
logs = sorted(glob.glob(log_path))
if logs:
    with open(logs[-1]) as f:
        lines = f.readlines()
    print("".join(lines[-30:]))  # last 30 lines
else:
    print("No training log found yet. Start training first.")

In [ ]:
# ── Plot training / validation loss curves ────────────────────────────────────
import json, glob
import matplotlib.pyplot as plt

CONFIG = "3d_fullres"
FOLD   = 0

progress_path = (
    f"{RESULTS}/{DATASET_NAME}/nnUNetTrainer__nnUNetPlans__{CONFIG}/"
    f"fold_{FOLD}/progress.png"
)
if os.path.exists(progress_path):
    from IPython.display import Image
    display(Image(progress_path))
else:
    print(f"Progress plot not found at:\n  {progress_path}")

---
## 5. Find Best Configuration

After all folds of each config finish, let nnUNet automatically determine the best single config and ensemble.

In [ ]:
# run(f"nnUNetv2_find_best_configuration {DATASET_ID} -c 2d 3d_fullres")

---
## 6. Inference (Prediction)

Run predictions on the held-out **test set** (`imagesTs/`) or any new folder of images.

In [ ]:
import os

CONFIG      = "3d_fullres"
FOLDS       = "0 1 2 3 4"      # space-separated; use a single fold for quick testing
INPUT_DIR   = f"{RAW}/{DATASET_NAME}/imagesTs"
OUTPUT_DIR  = f"{NNUNET}/predictions/{CONFIG}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

cmd = (
    f"nnUNetv2_predict"
    f" -i {INPUT_DIR}"
    f" -o {OUTPUT_DIR}"
    f" -d {DATASET_ID}"
    f" -c {CONFIG}"
    f" -f {FOLDS}"
    f" --save_probabilities"
)
print("Command to run:\n", cmd)
# run(cmd)

---
## 7. Evaluation

Compute Dice and other metrics between predictions and ground-truth masks.

In [ ]:
# ── nnUNet built-in evaluator ─────────────────────────────────────────────────
# Requires ground-truth labels in a separate folder.

GT_DIR   = f"{RAW}/{DATASET_NAME}/labelsTr"   # adjust to test labels if available
PRED_DIR = f"{NNUNET}/predictions/3d_fullres"

eval_cmd = (
    f"nnUNetv2_evaluate_folder"
    f" -ref {GT_DIR}"
    f" -pred {PRED_DIR}"
    f" -djfile {RAW}/{DATASET_NAME}/dataset.json"
    f" -pfile {PREPROCESSED}/{DATASET_NAME}/nnUNetPlans.json"
)
print("Command to run:\n", eval_cmd)
# run(eval_cmd)

In [ ]:
# ── Manual Dice computation ───────────────────────────────────────────────────
import nibabel as nib
import numpy as np
import os

def dice(pred, gt, label=1):
    p = (pred == label).astype(bool)
    g = (gt   == label).astype(bool)
    intersection = (p & g).sum()
    denom = p.sum() + g.sum()
    return 2 * intersection / denom if denom > 0 else float('nan')

# Example: compare one prediction against its ground-truth
PRED_DIR = f"{NNUNET}/predictions/3d_fullres"
GT_DIR   = f"{RAW}/{DATASET_NAME}/labelsTr"

dices = []
pred_files = sorted(os.listdir(PRED_DIR)) if os.path.exists(PRED_DIR) else []
for fname in pred_files:
    gt_path   = f"{GT_DIR}/{fname}"
    pred_path = f"{PRED_DIR}/{fname}"
    if not os.path.exists(gt_path):
        continue
    pred_data = nib.load(pred_path).get_fdata()
    gt_data   = nib.load(gt_path).get_fdata()
    d = dice(pred_data, gt_data, label=1)
    dices.append(d)

if dices:
    dices = np.array(dices)
    print(f"Dice (cyst) over {len(dices)} cases:")
    print(f"  Mean   : {np.nanmean(dices):.4f}")
    print(f"  Median : {np.nanmedian(dices):.4f}")
    print(f"  Std    : {np.nanstd(dices):.4f}")
    print(f"  Min    : {np.nanmin(dices):.4f}")
    print(f"  Max    : {np.nanmax(dices):.4f}")
else:
    print("No predictions found. Run inference first.")

---
## 8. Visualise Predictions

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import os, random

PRED_DIR    = f"{NNUNET}/predictions/3d_fullres"
IMAGE_DIR   = f"{RAW}/{DATASET_NAME}/imagesTr"
GT_DIR      = f"{RAW}/{DATASET_NAME}/labelsTr"

pred_files = sorted(os.listdir(PRED_DIR)) if os.path.exists(PRED_DIR) else []
if not pred_files:
    print("No predictions found.")
else:
    fname    = random.choice(pred_files)
    stem     = fname.replace(".nii.gz", "")
    img_path = f"{IMAGE_DIR}/{stem}_0000.nii.gz"

    img  = nib.load(img_path).get_fdata()  if os.path.exists(img_path) else None
    pred = nib.load(f"{PRED_DIR}/{fname}").get_fdata()
    gt   = nib.load(f"{GT_DIR}/{stem}.nii.gz").get_fdata()  if os.path.exists(f"{GT_DIR}/{stem}.nii.gz") else None

    mid_z = pred.shape[2] // 2
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    if img is not None:
        axes[0].imshow(img[:, :, mid_z].T, cmap="gray", origin="lower", vmin=-200, vmax=400)
    axes[0].set_title(f"CT — {stem}")
    axes[0].axis("off")

    if img is not None:
        axes[1].imshow(img[:, :, mid_z].T, cmap="gray", origin="lower", vmin=-200, vmax=400)
    axes[1].imshow(pred[:, :, mid_z].T, cmap="Blues", alpha=0.5, origin="lower")
    axes[1].set_title("Prediction")
    axes[1].axis("off")

    if gt is not None and img is not None:
        axes[2].imshow(img[:, :, mid_z].T, cmap="gray", origin="lower", vmin=-200, vmax=400)
        axes[2].imshow(gt[:, :, mid_z].T,   cmap="Reds",  alpha=0.4, origin="lower")
        axes[2].imshow(pred[:, :, mid_z].T, cmap="Blues", alpha=0.3, origin="lower")
        axes[2].set_title("GT (red) + Pred (blue) overlay")
    axes[2].axis("off")

    plt.suptitle(f"Case: {stem} | Dice: {dice(pred, gt):.3f}" if gt is not None else f"Case: {stem}")
    plt.tight_layout()
    plt.show()

---
## 9. Fix CAD Mask Header Mismatch (Optional Data Repair)

Many CAD cases have masks with zeroed-out NIfTI headers (origin 0,0,0 and positive direction) while the images have the original DICOM metadata (non-zero origin, often LPS negative direction). This causes origin/direction warnings during preprocessing but does **not** affect nnUNet training since it resamples into a canonical space.

If you need corrected masks for downstream tools (e.g. 3D Slicer, ITK-SNAP), run the cell below to copy the image affine onto each mask.

In [ ]:
# ── OPTIONAL: fix mask headers to match image headers ─────────────────────────
import nibabel as nib
import numpy as np
import os, re

IMG_DIR  = f"{DATA_DIR}/images"
MASK_DIR = f"{DATA_DIR}/masks"
OUT_DIR  = f"{DATA_DIR}/masks_fixed"
os.makedirs(OUT_DIR, exist_ok=True)

def img_stem_to_mask(stem):
    s = re.sub(r'([A-Za-z]+)(\d+)', lambda m: m.group(1).lower()+'_'+m.group(2), stem)
    return f"cyst_{s}.nii.gz"

fixed, skipped = 0, 0
for img_fname in sorted(os.listdir(IMG_DIR)):
    stem      = img_fname.replace(".nii.gz", "")
    mask_name = img_stem_to_mask(stem)
    mask_path = f"{MASK_DIR}/{mask_name}"
    if not os.path.exists(mask_path):
        skipped += 1
        continue

    img_nib  = nib.load(f"{IMG_DIR}/{img_fname}")
    mask_nib = nib.load(mask_path)

    # Only fix if direction is identity (all positive = zeroed-out header)
    direction = np.linalg.norm(mask_nib.affine[:3, :3], axis=0)
    if np.allclose(mask_nib.affine[:3, :3] / direction, np.eye(3), atol=0.01):
        # Copy image affine onto mask
        new_mask = nib.Nifti1Image(mask_nib.get_fdata(), img_nib.affine, img_nib.header)
        nib.save(new_mask, f"{OUT_DIR}/{mask_name}")
        fixed += 1
    else:
        # Direction already looks correct; just copy
        import shutil
        shutil.copy(mask_path, f"{OUT_DIR}/{mask_name}")

print(f"Fixed: {fixed} | Passed through: {skipped} | Output: {OUT_DIR}")

---
## 10. Quick Reference — All nnUNet CLI Commands

```bash
# Set environment (always required)
export nnUNet_raw=/home/huy/quan_nguyen/aima/pancrea_cyst/nnUnet/nnUNet_raw
export nnUNet_preprocessed=/home/huy/quan_nguyen/aima/pancrea_cyst/nnUnet/nnUNet_preprocessed
export nnUNet_results=/home/huy/quan_nguyen/aima/pancrea_cyst/nnUnet/nnUNet_results
export PATH=/home/huy/quan_nguyen/aima/pancrea_cyst/.venv/bin:$PATH

# 1. Plan + preprocess (already done)
nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity -np 8

# 2. Train (one fold at a time)
nnUNetv2_train 1 3d_fullres 0 --npz
nnUNetv2_train 1 3d_fullres 1 --npz
# ... folds 2, 3, 4

# 3. Resume interrupted training
nnUNetv2_train 1 3d_fullres 0 --npz -continue_training

# 4. Find best config (after all folds done)
nnUNetv2_find_best_configuration 1 -c 2d 3d_fullres

# 5. Predict test set
nnUNetv2_predict -i $nnUNet_raw/Dataset001_PancreasCyst/imagesTs \
                 -o $nnUNet_results/predictions/3d_fullres \
                 -d 1 -c 3d_fullres -f 0 1 2 3 4

# 6. Evaluate
nnUNetv2_evaluate_folder \
    -ref $nnUNet_raw/Dataset001_PancreasCyst/labelsTr \
    -pred $nnUNet_results/predictions/3d_fullres \
    -djfile $nnUNet_raw/Dataset001_PancreasCyst/dataset.json \
    -pfile $nnUNet_preprocessed/Dataset001_PancreasCyst/nnUNetPlans.json
```

### ResEncoder presets (recommended for new experiments)
nnUNet v2.6+ recommends the ResEncoder planner for better performance:
```bash
nnUNetv2_plan_and_preprocess -d 1 -pl nnUNetPlannerResEncM --verify_dataset_integrity
nnUNetv2_train 1 3d_fullres 0 -p nnUNetResEncUNetMPlans --npz
```